# Prism × nanoGPT: Shakespeare Self-Extraction
**Runtime → A100 or T4**

The right test: extract spectra from a **trained Shakespeare model**,
then use those spectra to initialize a **fresh Shakespeare model**.
Same architecture, same task, same data — the spectra actually match.

Pipeline:
1. Train baseline to convergence (5K steps, ~5 min)
2. Extract spectral fingerprint from converged model
3. Train fresh model with Prism init using extracted fingerprint
4. Compare convergence curves

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/timepointai/nanogpt-prism-shakespeare.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py
print('Ready.')

In [ ]:
# Cell 2: Train baseline to convergence (5K steps = ~5 min on A100)
# This gives us both the baseline numbers AND a trained model to extract from.
import subprocess, time

print('='*60)
print('  STEP 1: Train baseline to convergence (5K steps)')
print('='*60)

t0 = time.time()
result = subprocess.run([
    'python', 'train.py',
    'config/train_shakespeare_char.py',
    '--max_iters=5000',
    '--eval_interval=500',
    '--eval_iters=50',
    '--log_interval=100',
    '--out_dir=out-baseline-converged',
    '--always_save_checkpoint=True',
    '--wandb_log=False',
    '--compile=False',
    '--prism_init=False',
], capture_output=True, text=True, timeout=1200)

print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

with open('/content/baseline_converged.txt', 'w') as f:
    f.write(result.stdout)
print(f'\nDone in {time.time()-t0:.0f}s')
print('SAVED: /content/baseline_converged.txt')

In [ ]:
# Cell 3: Extract spectral fingerprint from converged model
import subprocess

print('='*60)
print('  STEP 2: Extract spectra from trained Shakespeare model')
print('='*60)

result = subprocess.run([
    'python', 'prism_extract.py',
    '--ckpt', 'out-baseline-converged/ckpt.pt',
    '--out', '.prism_cache/shakespeare',
], capture_output=True, text=True, timeout=120)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
print('Extraction complete.')

In [ ]:
# Cell 4: Train fresh model with Prism init (same 5K steps)
import subprocess, time

print('='*60)
print('  STEP 3: Train with Prism init from Shakespeare spectra')
print('='*60)

t0 = time.time()
result = subprocess.run([
    'python', 'train.py',
    'config/train_shakespeare_char.py',
    '--max_iters=5000',
    '--eval_interval=500',
    '--eval_iters=50',
    '--log_interval=100',
    '--out_dir=out-prism-shakespeare',
    '--wandb_log=False',
    '--compile=False',
    '--prism_init=True',
    '--prism_align=0.75',
    '--prism_spectra=.prism_cache/shakespeare/spectra.json',
    '--prism_directions=.prism_cache/shakespeare/directions.pt',
], capture_output=True, text=True, timeout=1200)

print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

with open('/content/prism_shakespeare.txt', 'w') as f:
    f.write(result.stdout)
print(f'\nDone in {time.time()-t0:.0f}s')
print('SAVED: /content/prism_shakespeare.txt')

In [ ]:
# Cell 5: Compare convergence curves
import re

def parse_losses(log_path):
    val = {}
    train = {}
    with open(log_path) as f:
        for line in f:
            m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
            if m:
                step = int(m.group(1))
                train[step] = float(m.group(2))
                val[step] = float(m.group(3))
    return train, val

b_train, b_val = parse_losses('/content/baseline_converged.txt')
p_train, p_val = parse_losses('/content/prism_shakespeare.txt')

print('='*60)
print('  RESULTS: Shakespeare Self-Extraction Test')
print('='*60)
print(f'\n{"Step":>6s}  {"Base val":>10s}  {"Prism val":>10s}  {"Diff":>8s}  {"Winner":>8s}')
print('-' * 50)

all_steps = sorted(set(b_val.keys()) | set(p_val.keys()))
for step in all_steps:
    b = b_val.get(step)
    p = p_val.get(step)
    if b and p:
        diff = b - p
        winner = 'PRISM' if p < b else 'BASE' if b < p else 'TIE'
        print(f'{step:>6d}  {b:>10.4f}  {p:>10.4f}  {diff:>+8.4f}  {winner:>8s}')

if b_val and p_val:
    last_step = max(set(b_val.keys()) & set(p_val.keys()))
    lb = b_val[last_step]
    lp = p_val[last_step]
    print(f'\nAt step {last_step}: baseline={lb:.4f}, prism={lp:.4f}')
    if lp < lb:
        print(f'Prism wins by {lb-lp:.4f} val loss')
        # Find when baseline reaches Prism\'s final val loss
        for s in sorted(b_val.keys()):
            if b_val[s] <= lp:
                print(f'Baseline reaches Prism\'s final loss at step {s} → Prism saves {last_step-s} steps')
                break
    else:
        print(f'Baseline wins by {lp-lb:.4f} val loss')

In [ ]:
# Cell 6: Download
from google.colab import files
for f in ['/content/baseline_converged.txt', '/content/prism_shakespeare.txt']:
    try: files.download(f)
    except: print(f'{f} not found')